# Multithreaded & parallel ML

The [intro](../intro.md) claims Rust parallelizes CPU work "for free" — no GIL,
compiled, threads that actually run at once. This chapter makes that concrete
with a **real benchmark** instead of an abstract claim, using
[`rayon`](https://docs.rs/rayon), Rust's data-parallelism library. Its core
trick: change `.iter()` to `.par_iter()` and a loop runs across all your cores.

```{note}
Timings vary run to run and with core count — treat the *ratio* as the signal,
not the absolute milliseconds.
```

In [ ]:
:dep rayon = { version = "1" }
:dep smartcore = { version = "0.3" }
// Warm up rayon on its own with a trivial parallel reduction.
{
    use rayon::prelude::*;
    let s: u64 = (0..1_000_000u64).into_par_iter().map(|x| x % 7).sum();
    println!("rayon ready (parallel sum = {})", s);
}

## Sequential vs. parallel — the one-line change

We fit a KNN model on 150 bootstrap resamples (an embarrassingly parallel task)
two ways: `.iter()` (one core) and `.into_par_iter()` (all cores). Identical
results, different wall-clock time. Everything lives in one block:

In [ ]:
{
    use rayon::prelude::*;
    use std::time::Instant;
    use smartcore::linalg::basic::matrix::DenseMatrix;
    use smartcore::api::SupervisedEstimator;
    use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
    use smartcore::metrics::accuracy;

    // Two separable-ish classes, 200 points, 2 features.
    let rows: Vec<Vec<f64>> = (0..200).map(|i| {
        let c = (i % 2) as f64 * 5.0;
        let j = (i / 2) as f64;
        vec![c + (j * 0.13).sin(), c + (j * 0.29).cos()]
    }).collect();
    let y: Vec<u32> = (0..200).map(|i| (i % 2) as u32).collect();

    // Fit KNN on a reproducible bootstrap resample (LCG), score on full data.
    let eval = |k: usize, seed: u64| -> f64 {
        let mut s = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
        let idx: Vec<usize> = (0..rows.len()).map(|_| {
            s = s.wrapping_mul(6364136223846793005).wrapping_add(1442695040888963407);
            (s >> 33) as usize % rows.len()
        }).collect();
        let br: Vec<Vec<f64>> = idx.iter().map(|&i| rows[i].clone()).collect();
        let by: Vec<u32> = idx.iter().map(|&i| y[i]).collect();
        let bx = DenseMatrix::new(br.len(), br[0].len(), br.iter().flatten().cloned().collect(), false);
        let m = KNNClassifier::fit(&bx, &by, KNNClassifierParameters::default().with_k(k)).unwrap();
        let fx = DenseMatrix::new(rows.len(), rows[0].len(), rows.iter().flatten().cloned().collect(), false);
        accuracy(&y, &m.predict(&fx).unwrap())
    };

    let t = Instant::now();
    let seq: Vec<f64> = (0..150u64).map(|s| eval(5, s)).collect();
    let seq_time = t.elapsed();

    let t = Instant::now();
    let par: Vec<f64> = (0..150u64).into_par_iter().map(|s| eval(5, s)).collect();
    let par_time = t.elapsed();

    println!("sequential: {:?}", seq_time);
    println!("parallel:   {:?}", par_time);
    println!("speedup:    {:.1}x", seq_time.as_secs_f64() / par_time.as_secs_f64());
    println!("identical results: {}", seq == par);
}

## Parallel grid search

The same idea applied to hyperparameter tuning: evaluate each candidate `k` for
KNN on its own core. This is the [optimization chapter's](../05b-optimization/hyperparameter-search.ipynb)
grid search, parallelized by swapping one iterator:

In [ ]:
{
    use rayon::prelude::*;
    use std::time::Instant;
    use smartcore::linalg::basic::matrix::DenseMatrix;
    use smartcore::api::SupervisedEstimator;
    use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
    use smartcore::metrics::accuracy;

    let rows: Vec<Vec<f64>> = (0..200).map(|i| {
        let c = (i % 2) as f64 * 5.0; let j = (i / 2) as f64;
        vec![c + (j * 0.13).sin(), c + (j * 0.29).cos()]
    }).collect();
    let y: Vec<u32> = (0..200).map(|i| (i % 2) as u32).collect();

    let score = |k: usize| -> f64 {
        let x = DenseMatrix::new(rows.len(), rows[0].len(), rows.iter().flatten().cloned().collect(), false);
        let m = KNNClassifier::fit(&x, &y, KNNClassifierParameters::default().with_k(k)).unwrap();
        accuracy(&y, &m.predict(&x).unwrap())
    };

    let ks = vec![3usize, 5, 7, 9, 11, 13, 15];
    let t = Instant::now();
    let results: Vec<(usize, f64)> = ks.par_iter().map(|&k| (k, score(k))).collect();
    println!("grid searched {} values in parallel in {:?}", ks.len(), t.elapsed());
    for (k, acc) in &results { println!("  k={:>2}  accuracy={:.3}", k, acc); }
    let best = results.iter().max_by(|a, b| a.1.partial_cmp(&b.1).unwrap()).unwrap();
    println!("best k = {} (accuracy {:.3})", best.0, best.1);
}

## When parallelism does — and doesn't — help

Parallelism is a win when each task is **substantial and independent**, as above.
It's *not* free on:

- **Tiny workloads** — spawning and coordinating threads has overhead; on a
  handful of microsecond tasks, the sequential version can actually win. If your
  speedup above is near or below 1x, that's this effect — worth reporting
  honestly rather than assuming parallel is always faster.
- **I/O-bound work** — waiting on disk/network isn't CPU-bound, so more threads
  don't help.

This mirrors the [intro's](../intro.md) "fair comparison" stance: Rust's parallel
advantage is real, but not universal.

```{note}
You often get this for free: `smartcore`'s `RandomForestClassifier`/`Regressor`
already use `rayon` internally to build their trees in parallel — the
[Ensemble chapter](../04c-ensemble/random-forests.ipynb) benefits from exactly
this without writing any `par_iter` yourself.
```

Next: [Ensemble & Forest Models](../04c-ensemble/random-forests.ipynb), where
combining many models — often in parallel — beats any single one.